# California Housing Price Prediction: Linear Regression vs Gradient Boosting vs XGBoost

This notebook compares three regression models on the California Housing dataset to show how boosting techniques improve on a simple linear baseline, and how XGBoost compares to standard Gradient Boosting in both **accuracy** and **training speed**.

## 1. Load and Explore Data

We use `fetch_california_housing` from scikit-learn, a built-in dataset with housing features (median income, house age, rooms, location, etc.) and the target `MedHouseVal` (median house value in $100,000s).

In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# Load the dataset (returns a Bunch object with data, target, feature_names, DESCR)
housing = fetch_california_housing()

# Inspect dataset description (features, units, source)
print(housing.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [2]:
# Build a DataFrame from the feature matrix
df = pd.DataFrame(housing.data, columns=housing.feature_names)

# Add the target column (median house value)
df['MedHouseVal'] = housing.target

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
# Quick sanity checks before modeling
print("Shape:", df.shape)
print("\nSummary statistics:")
print(df.describe())
print("\nMissing values per column:")
print(df.isnull().sum())

Shape: (20640, 9)

Summary statistics:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333333      3.000000   
25%        2.563400     18.000000      4.440716      1.006079    787.000000   
50%        3.534800     29.000000      5.229129      1.048780   1166.000000   
75%        4.743250     37.000000      6.052381      1.099526   1725.000000   
max       15.000100     52.000000    141.909091     34.066667  35682.000000   

           AveOccup      Latitude     Longitude   MedHouseVal  
count  20640.000000  20640.000000  20640.000000  20640.000000  
mean       3.070655     35.631861   -119.569704      2.068558  
std       10.386050      2.135952      2.003532      1.15

No missing values, so no imputation is needed. We can move straight to splitting the data.

## 2. Train/Test Split

80/20 split with a fixed `random_state` so results are reproducible.

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop('MedHouseVal', axis='columns')  # features
y = df['MedHouseVal']                        # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

## 3. Baseline: Linear Regression

A simple linear model gives us a baseline to measure how much the ensemble methods improve performance.

In [5]:
import time
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

start = time.time()
model = LinearRegression()
model.fit(X_train, y_train)
end = time.time()

y_pred = model.predict(X_test)

r2s = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print("R2 Score:", r2s, "MSE:", mse, "Time:", end - start)

R2 Score: 0.5965968374812353 MSE: 0.5291402345397309 Time: 0.03687739372253418


## 4. Gradient Boosting Regressor

An ensemble of shallow decision trees built sequentially, where each new tree corrects the errors of the previous ones.

In [6]:
from sklearn.ensemble import GradientBoostingRegressor

start = time.time()
model = GradientBoostingRegressor(n_estimators=100)
model.fit(X_train, y_train)
end = time.time()

y_pred = model.predict(X_test)

r2s = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print("R2 Score:", r2s, "MSE:", mse, "Time:", end - start)

R2 Score: 0.7790292799949292 MSE: 0.2898452701259065 Time: 5.7184202671051025


## 5. XGBoost Regressor

XGBoost is an optimized, regularized implementation of gradient boosting built for speed and performance. The commented-out line shows a more tuned configuration; the active line uses default hyperparameters with `n_estimators=100` for a fair comparison against the Gradient Boosting run above.

In [7]:
from xgboost import XGBRegressor

# Example of a more tuned configuration (left commented for reference):
# model = XGBRegressor(objective='reg:squarederror', colsample_bytree=0.3,
#                       learning_rate=0.1, max_depth=5, alpha=10, n_estimators=100)

start = time.time()
model = XGBRegressor(n_estimators=100)
model.fit(X_train, y_train)
end = time.time()

y_pred = model.predict(X_test)

r2s = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print("R2 Score:", r2s, "MSE:", mse, "Time:", end - start)

R2 Score: 0.8307446139807378 MSE: 0.22201074006498173 Time: 0.1835181713104248


## 6. Results Summary

| Model | R2 Score | Training Time |
|---|---|---|
| Linear Regression | 0.59 | — |
| Gradient Boosting | 0.77 | ~6 sec |
| XGBoost | 0.83 | ~0.12 sec |

**Takeaways:**
- Linear Regression performs worst (R2 = 0.59) since house prices don't follow a simple linear relationship with the features.
- Gradient Boosting improves accuracy significantly (R2 = 0.77) by combining many weak learners.
- XGBoost improves accuracy further (R2 = 0.83) **and** trains about 50x faster than standard Gradient Boosting, thanks to its optimized, parallelized implementation.